In [ ]:
import tilelang
import tilelang.language as T

tilelang.disable_cache()

M, N = 512, 512
M, N = 256, 256
block_M, block_N = 32, 32
dtype = "float32"

@tilelang.jit(out_idx=[-1], target="c", target_host="c", execution_backend="cython")
def my_add():
    @T.prim_func
    def main(
        A: T.Tensor((M, N), dtype), # type: ignore
        B: T.Tensor((M, N), dtype), # type: ignore
        C: T.Tensor((M, N), dtype), # type: ignore
    ):
        with T.Kernel(T.ceildiv(N, block_N), T.ceildiv(M, block_M)) as (bx, by):
            A_shared = T.alloc_shared((block_M, block_N), dtype, scope="shared")
            B_shared = T.alloc_shared((block_M, block_N), dtype, scope="shared")
            C_local = T.alloc_fragment((block_M, block_N), dtype)
            C_shared = T.alloc_shared((block_M, block_N), dtype, scope="shared")

            T.copy(A[by * block_M, bx * block_N], A_shared)
            T.copy(B[by * block_M, bx * block_N], B_shared)

            for i, j in T.Parallel(block_M, block_N):
                C_local[i, j] = A_shared[i, j] + B_shared[i, j]

            T.copy(C_local, C_shared)
            T.copy(C_shared, C[by * block_M, bx * block_N])

    return main


add_kernel = my_add()
print(add_kernel.get_kernel_source())


In [ ]:
import torch

torch.manual_seed(0)

a = torch.randn(M, N)
b = torch.randn(M, N)
c_golden = a + b

# print(f"Input tensor a: {a}")
# print(f"Input tensor b: {b}")
# print(f"Golden tensor c: {c_golden}")

c = add_kernel(a, b)
# print(f"Output tensor c: {c}")

torch.testing.assert_close(c, c_golden, rtol=1e-5, atol=1e-5)
print("Test passed: The output tensor c is equal to the sum of the input tensors a and b.")
